In [13]:
import pandas as pd

In [14]:
df = pd.read_csv('data-tbtl/annonimized.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 295198 entries, 0 to 295197
Data columns (total 11 columns):
 #   Column                           Non-Null Count   Dtype 
---  ------                           --------------   ----- 
 0   concat('it001',`assignment_id`)  295198 non-null  object
 1   concat('it001',`problem_id`)     295198 non-null  object
 2   concat('it001', username)        295198 non-null  object
 3   is_final                         295198 non-null  int64 
 4   status                           295198 non-null  object
 5   pre_score                        295198 non-null  int64 
 6   coefficient                      295198 non-null  int64 
 7   concat('it001',`language_id`)    295198 non-null  object
 8   created_at                       295198 non-null  object
 9   updated_at                       295198 non-null  object
 10  judgement                        295198 non-null  object
dtypes: int64(3), object(8)
memory usage: 24.8+ MB


In [15]:
# Calculate accepted status based on judgement
df['accepted'] = df['judgement'].apply(lambda x: 1 if '"verdicts":[]' in x else 0)

# Drop unnecessary columns
df = df.drop(columns=["concat('it001',`language_id`)", "updated_at"])

df = df.rename(columns={
    "concat('it001',`problem_id`)": "problem_id",
    "concat('it001', username)": "username",
    "concat('it001',`assignment_id`)" : "assignment_id",
})


df['created_at'] = pd.to_datetime(df['created_at'], 
                                format='%m-%d %H:%M:%S',  # Specify format
                                errors='coerce')             # Handle errors gracefully

min_dates = df.groupby(['problem_id', 'assignment_id'])['created_at'].transform('min')

df['created_at'] = (df['created_at'] - min_dates).dt.total_seconds() / 3600

# Convert score to percentage
df['pre_score'] = df['pre_score'] / 10000 * 100

# Set score to 0 for non-accepted submissions
df.loc[df['accepted'] == 0, 'pre_score'] = 0

# Group by username and problem_id and calculate the sum of scores
grouped_df = df.groupby(['username', 'problem_id', 'assignment_id']).agg({'pre_score': 'max', 'accepted': 'max', 'created_at' : 'max'}).reset_index()

# Create a total time try for each problem id for each person
grouped_df['total_time_try'] = df.groupby(['username', 'problem_id']).size().reset_index(name='total_time_try')['total_time_try']

df = grouped_df
df['percent'] = df['accepted'] / df['total_time_try']
# Concatenate problem_id and assignment_id
# df['problem_id'] = df['problem_id'].astype(str) + '_' + df['assignment_id'].astype(str)

# Drop assignment_id column
df = df.drop(columns=['assignment_id'])
df.head()

,username,problem_id,pre_score,accepted,created_at,total_time_try,percent
0,00b6dd4fc7eb817e03708c532016ef30ce564a61,008f8e9b0f4fac6b5f188f0dc8d118a8b19aabee,100.0,1,0.828889,1.0,1.000000
1,00b6dd4fc7eb817e03708c532016ef30ce564a61,10ef40cac132f24608f893c45f1452173e00e0b6,100.0,1,0.620833,1.0,1.000000
2,00b6dd4fc7eb817e03708c532016ef30ce564a61,11cff2e1cb6f229a49b9be40225764aa6a38508f,100.0,1,173.191111,1.0,1.000000
3,00b6dd4fc7eb817e03708c532016ef30ce564a61,1d824e02c40fecdae25d89c0357410997dfbf799,100.0,1,1.511389,1.0,1.000000
4,00b6dd4fc7eb817e03708c532016ef30ce564a61,217ba7563116dab9ca8bf00875e14cec255d7525,100.0,1,154.305000,7.0,0.142857


In [16]:
# Group by username and calculate the sum of scores and total_time_try
grouped_by_username_df = df.groupby('username').agg({
    'pre_score': ['max', 'min', 'mean', 'median', 'sum'],
    'accepted': ['max', 'min', 'mean', 'median', 'sum'],
    'created_at': ['max', 'min', 'mean', 'median', 'sum'],
    'total_time_try': ['max', 'min', 'mean', 'median', 'sum'],
    'percent': ['max', 'min', 'mean', 'median', 'sum']
}).reset_index()
grouped_by_username_df.columns = ['_'.join(col).strip() for col in grouped_by_username_df.columns.values]
grouped_by_username_df.reset_index()
grouped_by_username_df = grouped_by_username_df.rename(columns={
    'username_': 'username',
})
grouped_by_username_df.head()

,username,pre_score_max,pre_score_min,pre_score_mean,pre_score_median,pre_score_sum,accepted_max,accepted_min,accepted_mean,accepted_median,...,total_time_try_max,total_time_try_min,total_time_try_mean,total_time_try_median,total_time_try_sum,percent_max,percent_min,percent_mean,percent_median,percent_sum
0,00b6dd4fc7eb817e03708c532016ef30ce564a61,100.0,100.0,100.000000,100.0,4700.0,1,1,1.000000,1.0,...,11.0,1.0,3.191489,2.0,150.0,1.0,0.090909,0.551242,0.50,25.908369
1,00bef8afee8f3c595d535c9c03c490cac1a4f021,100.0,0.0,92.307692,100.0,7200.0,1,0,0.923077,1.0,...,19.0,1.0,3.448718,2.0,269.0,1.0,0.000000,0.503812,0.50,39.297351
2,01122b3ef7e59b84189e65985305f575d6bdf83c,100.0,0.0,86.764706,100.0,5900.0,1,0,0.867647,1.0,...,27.0,1.0,2.794118,1.0,190.0,1.0,0.000000,0.622148,1.00,42.306085
3,0134f9f410c65ad0e8c2254a7e9288670e02a183,100.0,100.0,100.000000,100.0,4700.0,1,1,1.000000,1.0,...,11.0,1.0,2.085106,1.0,98.0,1.0,0.090909,0.722383,1.00,33.952020
4,013de369c439ab0ead8aa7da64423aa395a8be39,100.0,0.0,87.878788,100.0,5800.0,1,0,0.878788,1.0,...,13.0,1.0,2.166667,1.0,143.0,1.0,0.000000,0.627596,0.75,41.421368


In [17]:
df = grouped_by_username_df
th = pd.read_csv('public_it001/th-public.csv')
th = th.rename(columns={
    "hash": "username",
})
th.head()

,username,TH
0,00b6dd4fc7eb817e03708c532016ef30ce564a61,5
1,00bef8afee8f3c595d535c9c03c490cac1a4f021,8.5
2,01122b3ef7e59b84189e65985305f575d6bdf83c,7
3,013de369c439ab0ead8aa7da64423aa395a8be39,10
4,014c59c6433fd764a0b08de6ffeb757eaf60aa73,6


In [18]:
test = pd.merge(df, th, on='username', how='left', indicator=True)
test = test[test['_merge'] == 'left_only'].drop(columns=['_merge'])
test.head()

,username,pre_score_max,pre_score_min,pre_score_mean,pre_score_median,pre_score_sum,accepted_max,accepted_min,accepted_mean,accepted_median,...,total_time_try_min,total_time_try_mean,total_time_try_median,total_time_try_sum,percent_max,percent_min,percent_mean,percent_median,percent_sum,TH
3,0134f9f410c65ad0e8c2254a7e9288670e02a183,100.0,100.0,100.000000,100.0,4700.0,1,1,1.000000,1.0,...,1.0,2.085106,1.0,98.0,1.0,0.090909,0.722383,1.00,33.952020,NaN
20,035f97702f2c01d26ab1fae8f39ea2f98a0caa3c,100.0,100.0,100.000000,100.0,5000.0,1,1,1.000000,1.0,...,1.0,3.260000,2.0,163.0,1.0,0.083333,0.550842,0.50,27.542100,NaN
68,0aaebc88f6106684d6993c156104c1ef36cf94e0,100.0,100.0,100.000000,100.0,5000.0,1,1,1.000000,1.0,...,1.0,2.020000,2.0,101.0,1.0,0.166667,0.667333,0.50,33.366667,NaN
80,0bf111a9caedf02804f6991792490e63bc21058a,100.0,0.0,96.341463,100.0,7900.0,1,0,0.963415,1.0,...,1.0,3.378049,1.0,277.0,1.0,0.000000,0.638084,0.75,52.322917,NaN
120,12887fd9a4df4ba9b88a71f3fb1d2502a75995dd,100.0,0.0,98.437500,100.0,6300.0,1,0,0.984375,1.0,...,1.0,2.328125,1.0,149.0,1.0,0.000000,0.820449,1.00,52.508766,NaN


In [19]:
import numpy as np
merged_df = pd.merge(th, df, on='username', how='inner')
merged_df['TH'] = (merged_df['TH']
                   .replace(['\xa0', ''], np.nan)  # Replace invalid values with NaN
                   .astype(float))                 # Convert to numeric

# Calculate mean and fill NaN values
mean_th = merged_df['TH'].mean()
merged_df['TH'] = merged_df['TH'].fillna(mean_th)
# Display first few rows
df = merged_df
df.head()

,username,TH,pre_score_max,pre_score_min,pre_score_mean,pre_score_median,pre_score_sum,accepted_max,accepted_min,accepted_mean,...,total_time_try_max,total_time_try_min,total_time_try_mean,total_time_try_median,total_time_try_sum,percent_max,percent_min,percent_mean,percent_median,percent_sum
0,00b6dd4fc7eb817e03708c532016ef30ce564a61,5.0,100.0,100.0,100.000000,100.0,4700.0,1,1,1.000000,...,11.0,1.0,3.191489,2.0,150.0,1.0,0.090909,0.551242,0.50,25.908369
1,00bef8afee8f3c595d535c9c03c490cac1a4f021,8.5,100.0,0.0,92.307692,100.0,7200.0,1,0,0.923077,...,19.0,1.0,3.448718,2.0,269.0,1.0,0.000000,0.503812,0.50,39.297351
2,01122b3ef7e59b84189e65985305f575d6bdf83c,7.0,100.0,0.0,86.764706,100.0,5900.0,1,0,0.867647,...,27.0,1.0,2.794118,1.0,190.0,1.0,0.000000,0.622148,1.00,42.306085
3,013de369c439ab0ead8aa7da64423aa395a8be39,10.0,100.0,0.0,87.878788,100.0,5800.0,1,0,0.878788,...,13.0,1.0,2.166667,1.0,143.0,1.0,0.000000,0.627596,0.75,41.421368
4,014c59c6433fd764a0b08de6ffeb757eaf60aa73,6.0,100.0,0.0,87.777778,100.0,7900.0,1,0,0.877778,...,17.0,1.0,2.711111,1.0,244.0,1.0,0.000000,0.592996,0.50,53.369610


In [20]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

numeric_features = df.select_dtypes(include=[np.number])
X = numeric_features.drop(['TH', 'username'] if 'username' in numeric_features.columns else ['TH'], axis=1)
y = df['TH']

# Normalize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

In [21]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_regression

# Assuming X_scaled is already scaled
X = X_scaled  
y = df['TH']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create polynomial features
poly = PolynomialFeatures(degree=1, include_bias=True)

# Feature selection
select_k_best = SelectKBest(score_func=f_regression, k=10)

# Dimensionality reduction
pca = PCA(n_components=10)

# Create pipelines for each model
rf =  RandomForestRegressor(
    n_estimators=1000, 
    max_depth=5, 
    max_features='sqrt',
    random_state=42
)

gb = GradientBoostingRegressor(
    n_estimators=1000,  
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42
)

# Create Voting Regressor
voting_regressor = VotingRegressor(estimators=[
    ('rf', rf),
    ('gb', gb)
])

# Fit the model using cross-validation
cv_scores = cross_val_score(voting_regressor, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
print(f'Cross-validated MSE: {-cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

# Fit the model on the entire training set
voting_regressor.fit(X_train, y_train)

# Make predictions
predictions = voting_regressor.predict(X_test)
train_pred = voting_regressor.predict(X_train)

# Calculate metrics
mse = mean_squared_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

mse_train = mean_squared_error(y_train, train_pred)
r2_train = r2_score(y_train, train_pred)

# Output results
print(f'Voting Regressor with Random Forest and Gradient Boosting')
print(f'Mean Squared Error: {mse:.4f}')
print(f'R2 Score: {r2:.4f}')
print(f'Mean Squared Error Train: {mse_train:.4f}')
print(f'R2 Score Train: {r2_train:.4f}')

Cross-validated MSE: 3.2919 ± 0.6189
Voting Regressor with Random Forest and Gradient Boosting
Mean Squared Error: 3.5764
R2 Score: 0.3546
Mean Squared Error Train: 0.6749
R2 Score Train: 0.8495


In [22]:
numeric_features = test.select_dtypes(include=[np.number])
X = numeric_features
y = df['TH']
X = X.fillna(X.mean())

# Normalize features
#X.drop(columns=['TH'], inplace=True)
X_scaled = scaler.transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

test['TH'] = voting_regressor.predict(X_scaled)
test = test[['username', 'TH']]
test.head()

,username,TH
3,0134f9f410c65ad0e8c2254a7e9288670e02a183,5.542118
20,035f97702f2c01d26ab1fae8f39ea2f98a0caa3c,5.273329
68,0aaebc88f6106684d6993c156104c1ef36cf94e0,5.236116
80,0bf111a9caedf02804f6991792490e63bc21058a,8.993696
120,12887fd9a4df4ba9b88a71f3fb1d2502a75995dd,7.646692


In [23]:
test.to_csv('updated_test.csv', index=False)